# multiply-back composite — cx16: register multiply_back0 / multiply_back1 in the lookup

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 3 atoms together: `backward-func-lookup`, `multiply-back`, `arg-position-back-functions`
> Running the final beacon reports progress against all 3 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "multiply-back"
DD_ATOM_IDS = ["backward-func-lookup", "multiply-back", "arg-position-back-functions"]
DD_SUBTOPICS = ["Backprop: BackwardFuncLookup", "Backprop: multiply_back", "Backprop: Arg-position back funcs"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing multiply_back (with unbroadcast) into the BackwardFuncLookup

`multiply` is mathematically symmetric (`x*y = y*x`), but the manual autograd still registers TWO back fns — `multiply_back0` for arg-0 and `multiply_back1` for arg-1 — because the dispatcher doesn't know any op is symmetric. It just looks up `(fn, argnum)`.

Both bodies follow the SAME pattern: local derivative * grad_out, then `unbroadcast(...)` to collapse any broadcast axes back to the parent's shape. Then both register into the `BackwardFuncLookup` under `t.multiply` at argnum 0 and 1.

This composite has you wire the full registration end-to-end: write the back fns, register them, and then dispatch by `(t.multiply, argnum)` to compute grads.

### Composite Exercise — register multiply_back0 / multiply_back1 in the lookup

**Atoms exercised together**: `backward-func-lookup`, `multiply-back`, `arg-position-back-functions`

Implement:

**1. `BackwardFuncLookup`** with `add_back_func` and `get_back_func` (raises `KeyError` on miss).

**2. `multiply_back0(grad_out, out, x, y)`** — returns `unbroadcast(grad_out * y, x)`, shaped like `x`.

**3. `multiply_back1(grad_out, out, x, y)`** — returns `unbroadcast(grad_out * x, y)`, shaped like `y`.

**4. `cx16_build_lookup()`** — returns a populated `BackwardFuncLookup` with both back fns registered under `t.multiply` at argnum 0 and 1 (use `add_back_func`, not direct dict access).

The `unbroadcast(grad, original)` helper is provided. The test exercises lookup + dispatch + value + broadcast collapse.

In [ ]:
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}
    def add_back_func(self, forward_fn, arg_position, back_fn):
        self.back_funcs[(forward_fn, arg_position)] = back_fn
    def get_back_func(self, forward_fn, arg_position):
        key = (forward_fn, arg_position)
        if key not in self.back_funcs:
            raise KeyError(
                f'No back_fn for ({forward_fn!r}, argnum={arg_position}).'
            )
        return self.back_funcs[key]

def multiply_back0(grad_out, out, x, y):
    if not isinstance(y, t.Tensor):
        y = t.tensor(y)
    return unbroadcast(grad_out * y, x)

def multiply_back1(grad_out, out, x, y):
    if not isinstance(x, t.Tensor):
        x = t.tensor(x)
    return unbroadcast(grad_out * x, y)

def cx16_build_lookup():
    bf = BackwardFuncLookup()
    bf.add_back_func(t.multiply, 0, multiply_back0)
    bf.add_back_func(t.multiply, 1, multiply_back1)
    return bf


<details><summary>Show solution — cx16</summary>

```python
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}
    def add_back_func(self, forward_fn, arg_position, back_fn):
        self.back_funcs[(forward_fn, arg_position)] = back_fn
    def get_back_func(self, forward_fn, arg_position):
        key = (forward_fn, arg_position)
        if key not in self.back_funcs:
            raise KeyError(
                f'No back_fn for ({forward_fn!r}, argnum={arg_position}).'
            )
        return self.back_funcs[key]

def multiply_back0(grad_out, out, x, y):
    if not isinstance(y, t.Tensor):
        y = t.tensor(y)
    return unbroadcast(grad_out * y, x)

def multiply_back1(grad_out, out, x, y):
    if not isinstance(x, t.Tensor):
        x = t.tensor(x)
    return unbroadcast(grad_out * x, y)

def cx16_build_lookup():
    bf = BackwardFuncLookup()
    bf.add_back_func(t.multiply, 0, multiply_back0)
    bf.add_back_func(t.multiply, 1, multiply_back1)
    return bf
```

Symmetric ops still register twice — mirror bodies, separate (fn, argnum) keys. The dispatcher is the audience: it looks up by argnum regardless of the op's mathematical symmetry.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 3 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx16'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx16',
        'subtopics': ["Backprop: BackwardFuncLookup", "Backprop: multiply_back", "Backprop: Arg-position back funcs"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()